In [0]:
from dbr_py_utils.encryption_utils import AESEncryptor
from dbr_py_utils.aux_functions import df_string_field_curation, table_record_count_audit
from pyspark.sql import functions as F

In [0]:
%run ./Initial

In [0]:
expedia_bronze_table = "bronze.expedia_raw"
hotel_weather_bronze_table = "bronze.hotel_weather_raw"

In [0]:
expedia_bronze_df = spark.read.table(expedia_bronze_table)
hotel_weather_bronze_df = spark.read.table(hotel_weather_bronze_table)
logger.info(f"Read tables {expedia_bronze_table} and {hotel_weather_bronze_table}")

In [0]:
aes_key = dbutils.secrets.get("key-vault", "aes-encryption-key")
hotel_weather_encryptor = AESEncryptor(key=aes_key, attribute_list=["name", "address"])
hotel_weather_bronze_df = hotel_weather_bronze_df.transform(lambda df: hotel_weather_encryptor.aesDecrypt(df))
logger.info(f"Decrypted {hotel_weather_bronze_table}")

In [0]:
expedia_bronze_df = df_string_field_curation(expedia_bronze_df)
logger.info(f"Applied string field curation to {expedia_bronze_table}")
hotel_weather_bronze_df = df_string_field_curation(hotel_weather_bronze_df)
logger.info(f"Applied string field curation to {hotel_weather_bronze_table}")

In [0]:
expedia_bronze_df = expedia_bronze_df.select(
        F.col("id"),
        F.to_timestamp(F.col("date_time"), "yyyy-MM-dd HH:mm:ss").alias("date_time"),
        F.col("site_name"),
        F.col("posa_continent"),
        F.col("user_location_country"),
        F.col("user_location_region"),
        F.col("user_location_city"),
        F.col("orig_destination_distance"),
        F.col("user_id"),
        F.col("is_mobile"),
        F.col("is_package"),
        F.col("channel"),
        F.to_date(F.col("srch_ci"), "yyyy-MM-dd").alias("srch_ci"),
        F.to_date(F.col("srch_co"), "yyyy-MM-dd").alias("srch_co"),
        F.col("srch_adults_cnt"),
        F.col("srch_children_cnt"),
        F.col("srch_rm_cnt"),
        F.col("srch_destination_id"),
        F.col("srch_destination_type_id"),
        F.col("hotel_id"),
        F.lit(job_task_timestamp_utc).cast("timestamp").alias("insert_timestamp_utc")
    )

expedia_bronze_df = expedia_bronze_df.dropna(how="any", subset=expedia_bronze_df.columns).dropDuplicates(["id", "hotel_id"])

In [0]:
hotel_weather_bronze_df = hotel_weather_bronze_df.select(
    F.col("id").cast("bigint").alias("id"),
    F.col("address"),
    F.col("avg_tmpr_c").alias("average_temperature_celsius"),
    F.col("avg_tmpr_f").alias("average_temperature_fahrenheit"),
    F.col("city"),
    F.col("country"),
    F.col("geoHash"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("name"),
    F.to_date(F.col("wthr_date"), "yyyy-MM-dd").alias("weather_date"),
    F.lit(job_task_timestamp_utc).cast("timestamp").alias("insert_timestamp_utc"),
    F.col("year"),
    F.col("month"),
    F.col("day"),
)

hotel_weather_bronze_df = hotel_weather_bronze_df.dropna(how="any", subset=hotel_weather_bronze_df.columns).dropDuplicates(["id", "weather_date"])

In [0]:
hotel_weather_bronze_df = hotel_weather_bronze_df.transform(lambda df: hotel_weather_encryptor.aesEncrypt(df))
logger.info(f"Encrypted {hotel_weather_bronze_table}")

In [0]:
expedia_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"silver.expedia_processed")

logger.info(f"Persisted with overwrite processed expedia_bronze to silver.expedia_processed")
table_record_count_audit(spark, "silver.expedia_processed", job_task_timestamp_utc)

In [0]:
hotel_weather_bronze_df.write \
    .format("delta") \
    .partitionBy("year", "month", "day") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"silver.hotel_weather_processed")

logger.info(f"Persisted with overwrite processed hotel_weather_bronze to silver.hotel_weather_processed")
table_record_count_audit(spark, "silver.hotel_weather_processed", job_task_timestamp_utc)